# SimPO Training (Kaggle 2xT4)

Trains a SimPO (Simple Preference Optimization, Meng et al. 2024, princeton-nlp/SimPO) LoRA adapter on the exact same pairs file as the existing Full DPO run (`experiments/001_500_reasoning/data/dpo_pairs.jsonl`) -- training method is the only variable being tested here, not data.

SimPO differs from DPO in two ways, both reimplemented in `src/chart_prm/simpo/`: (1) it's reference-free -- the reward is the policy's own length-normalized log-probability, not a KL term against a frozen reference, so training needs half the forward passes per step that this project's DPO trainer does (no reference-model forward at all); (2) that length normalization (mean log-probability per response token, not summed) removes DPO's structural bias toward longer chosen responses. `beta` is much larger than DPO's (2.0 vs ~0.1) per the reference implementation's own tuning guidance.


In [ ]:
# Cell 1: Check GPU hardware and install sm_60 compatible PyTorch stack if Tesla P100 is assigned
# (identical to the proven-working kaggle_train_dpo/train_dpo.ipynb cell 1 -- no changes)
import os, subprocess, sys, torch

print(f'Initial PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    cc = torch.cuda.get_device_capability(0)
    print(f'GPU: {props.name}, Compute Capability: {cc}, VRAM: {props.total_memory / 1e9:.1f} GB')
    if cc[0] < 7:
        print(f'*** Tesla P100 (cc {cc}) detected. PyTorch 2.12 dropped sm_60 CUDA kernels.')
        print('*** Installing PyTorch 2.5.1+cu124 with full sm_60 CUDA GPU support...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
packages = ['transformers==4.49.0', 'peft==0.14.0', 'accelerate==1.2.1', 'qwen-vl-utils==0.0.14', 'pillow']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

import torch, transformers
print(f'Active PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU device: {torch.cuda.get_device_name(0)}')
print('Packages successfully configured.')


In [ ]:
# Cell 2: Checkout repository and execute SimPO trainer
import os, subprocess, sys
from pathlib import Path

repo_dir = Path('/tmp/chart-prm')
if repo_dir.exists():
    subprocess.run(['rm', '-rf', str(repo_dir)], check=True)

subprocess.run(['git', 'clone', 'https://github.com/yahorlahunovich/chart-prm.git', str(repo_dir)], check=True)
# main now has all Phase 1-3 DG-PRM code plus src/chart_prm/simpo/ -- plain clone defaults to main.
os.chdir(repo_dir)
print(f'Working directory set to {repo_dir}')

pairs_path = repo_dir / 'experiments/001_500_reasoning/data/dpo_pairs.jsonl'
assert pairs_path.exists(), f'Missing {pairs_path} -- commit/push problem.'
with open(pairs_path, encoding='utf-8') as f:
    n = sum(1 for line in f if line.strip())
print(f'Confirmed {n} DPO pairs present (same file Full DPO trained on).')

# Run SimPO training -- reference-free, so no reference-model plumbing at all.
env = os.environ.copy()
env['PYTHONPATH'] = 'src'
cmd = [
    sys.executable, 'scripts/train/train_simpo.py',
    '--dataset-path', 'experiments/001_500_reasoning/data/dpo_pairs.jsonl',
    '--output-dir', '/kaggle/working/qwen_vl_simpo_adapter',
    '--epochs', '1',
    '--batch-size', '1',
    '--lr', '1e-6',
    '--beta', '2.0',
    '--gamma-beta-ratio', '0.5'
]
subprocess.run(cmd, env=env, check=True)


In [ ]:
# Cell 3: Validate output artifacts
out_dir = Path('/kaggle/working/qwen_vl_simpo_adapter')
files = sorted([p.name for p in out_dir.iterdir()]) if out_dir.exists() else []
print(f'Adapter directory {out_dir} contents: {files}')
assert (out_dir / 'adapter_config.json').exists(), 'adapter_config.json missing -- training did not save a LoRA adapter.'
